<a href="https://colab.research.google.com/github/lisetperez-cmd/AI4ENG_2025-2_Entrega2_PerezLiset_DelCastilloMonica/blob/main/99_modelo_soluci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
train = pd.read_csv("train.csv")
data = train.copy()

In [3]:
#Preprocesado del Data
#Imputación para columnas numéricas
#Esta función transforma valores categóricos
#relacionados con los rangos de matrícula universitaria a valores numéricos promedio.

def valor_matricula(valor):
    if pd.isna(valor):
        return np.nan
    elif 'Menos de 500 mil' in valor:
        return 250000  # Asignamos un valor promedio entre 0 y 500 mil
    elif 'Entre 500 mil y menos de 1 millón' in valor:
        return 750000  # Promedio entre 500 mil y 1 millón
    elif 'Entre 1 millón y menos de 2.5 millones' in valor:
        return 1750000  # Promedio entre 1 y 2.5 millones
    elif 'Entre 2.5 millones y menos de 4 millones' in valor:
        return 3250000  # Promedio entre 2.5 y 4 millones
    elif 'Entre 4 millones y menos de 5.5 millones' in valor:
        return 4750000  # Promedio entre 4 y 5.5 millones
    elif 'Entre 5.5 millones y menos de 7 millones' in valor:
        return 6250000  # Promedio entre 5.5 y 7 millones
    elif 'Más de 7 millones' in valor:
        return 7500000  # Asignamos un valor mínimo representativo superior a 7 millones
    elif 'No pagó matrícula' in valor:
        return 0  # Asumimos que no se pagó nada
    else:
        return np.nan  # Para cualquier caso que no coincida

In [4]:
#Esta función transforma valores categóricos que representan rangos
#de horas trabajadas semanalmente en valores numéricos promedio o representativos.
def horas_trabajadas(valor):
    if isinstance(valor, str):
        if "Entre" in valor:
            partes = valor.split('y')
            min_val = float(partes[0].split(' ')[-2])  # Obtener el penúltimo elemento
            max_val = float(partes[1].split(' ')[-2])  # Obtener el penúltimo elemento de la segunda parte
            return (min_val + max_val) / 2  # Retornar el promedio del rango

        elif "Más de" in valor:
            return float(valor.split(' ')[2])  # Convertir a número

        elif "Menos de" in valor:
            return float(valor.split(' ')[2])  # Convertir a número

        elif "0" in valor:
            return 0  # Devolver 0 en número

    return np.nan  # Devolver NaN si no es un valor válido

In [5]:
import pandas as pd
import numpy as np

def valor_matricula(valor):
    # Caso 1: nulos
    if pd.isna(valor):
        return np.nan

    # Convertir siempre a string para evitar el TypeError
    valor = str(valor)

    if 'Menos de 500 mil' in valor:
        return 250000
    elif 'Entre 500 mil y menos de 1 millón' in valor:
        return 750000
    elif 'Entre 1 millón y menos de 2 millones' in valor:
        return 1500000
    elif '2 millones o más' in valor:
        return 2500000

    # Si no coincide con ningún patrón, devolver NaN o valor original
    return np.nan

In [6]:
#Aplicación de funciones e imputaciones para columnas numéricas
# Aplicar las funciones de conversión
data['E_VALORMATRICULAUNIVERSIDAD'] = data['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)

data['E_HORASSEMANATRABAJA'] = data['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

# Imputar valores faltantes
num_imputer = SimpleImputer(strategy='mean')

data[['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']] = num_imputer.fit_transform(
    data[['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']]
)

# Agregar num_columns
num_columns = data.select_dtypes(include=['number']).columns
num_columns = [col for col in num_columns if col not in ['ID']]

In [8]:
#Imputación para columnas categóricas
cat_columns = ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'F_ESTRATOVIVIENDA',
               'F_TIENEINTERNET', 'E_PAGOMATRICULAPROPIO']

cat_imputer = SimpleImputer(strategy='most_frequent')  # Usamos la moda para imputar valores categóricos


data[cat_columns] = cat_imputer.fit_transform(data[cat_columns])

In [9]:
#######
#####Conversión de variables categóricas

binary_columns = ['F_TIENEINTERNET', 'E_PAGOMATRICULAPROPIO']


def map_binary_columns(data, binary_columns, mapping={'Si': 1, 'No': 0}):
    for col in binary_columns:
        data[col] = data[col].map(mapping)
    return data

data = map_binary_columns(data, binary_columns)

In [10]:
#Codificación de variables categóricas multiclase con diccionario

# Crear un diccionario para asignar el valor numérico a cada valor en letra de la columna FAMI_ESTRATOVIVIENDA
moda_estrato = data['F_ESTRATOVIVIENDA'].mode()[0]

estrato_dict = {
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6,
    'Sin Estrato': 0,
    np.nan: 0
}

data['F_ESTRATOVIVIENDA'] = data['F_ESTRATOVIVIENDA'].replace(estrato_dict)

/tmp/ipython-input-1760501436.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['F_ESTRATOVIVIENDA'] = data['F_ESTRATOVIVIENDA'].replace(estrato_dict)


In [11]:
#Esta celda transforma los valores categóricos en la columna RENDIMIENTO_GLOBAL, que representan niveles de rendimiento
#(por ejemplo, 'bajo', 'alto'), en valores numéricos mediante un mapeo definido en un diccionario.
rendimiento_mapping = {'bajo': 0, 'medio-bajo': 1, 'medio-alto': 2, 'alto': 3}
data['RENDIMIENTO_GLOBAL'] = data['RENDIMIENTO_GLOBAL'].map(rendimiento_mapping)

In [12]:
#Esta celda aplica codificación de etiquetas (Label Encoding) a las columnas categóricas especificadas
#en la lista label_columns. Este proceso convierte los valores categóricos en números enteros secuenciales.
#Adicionalmente, almacena los codificadores (LabelEncoder) utilizados, permitiendo reutilizarlos posteriormente
#para transformar datos nuevos o invertir el proceso.


label_columns = ['E_PRGM_ACADEMICO']

def apply_label_encoding(data, columns, label_encoders={}):
    for col in columns:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])
        label_encoders[col] = le
    return data, label_encoders

data, label_encoders = apply_label_encoding(data, label_columns)


In [13]:
#One-Hot Encoding para variables indicadoras (educación de padres y madres)

columns_to_encode = ['F_EDUCACIONMADRE','F_EDUCACIONPADRE']

def apply_onehot_encoding(data, columns, fill_value='No Aplica'):
  onehot_encoders = {}

  for col in columns:
        data[col].fillna(fill_value, inplace=True)
        unique_values = sorted(data[col].unique())
        onehot_encoders[col] = unique_values
        onehot_data = pd.get_dummies(data[col], prefix=col)
        data = pd.concat([data, onehot_data], axis=1)
        data.drop(col, axis=1, inplace=True)
  return data, onehot_encoders

data, onehot_encoders = apply_onehot_encoding(data, columns_to_encode)

/tmp/ipython-input-915640839.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(fill_value, inplace=True)
/tmp/ipython-input-915640839.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.

In [14]:
#Escalado de variables numéricas
scaler = MinMaxScaler(feature_range=(-1, 1))
data[num_columns] = scaler.fit_transform(data[num_columns])

data['E_PRGM_ACADEMICO'] = scaler.fit_transform(data[['E_PRGM_ACADEMICO']])
data['F_ESTRATOVIVIENDA'] = scaler.fit_transform(data[['F_ESTRATOVIVIENDA']])


In [15]:
#Guardamos el preprocesamiento en un diccionario para usarlo en el conjunto de prueba

preprocessing_objects = {
    'num_imputer': num_imputer,
    'cat_imputer': cat_imputer,
    'label_encoders': label_encoders,
    'onehot_encoders': onehot_encoders,
}

In [16]:
#Dataset preprocesado
data

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_TIENELAVADORA,F_TIENEAUTOMOVIL,...,F_EDUCACIONPADRE_Ninguno,F_EDUCACIONPADRE_No Aplica,F_EDUCACIONPADRE_No sabe,F_EDUCACIONPADRE_Postgrado,F_EDUCACIONPADRE_Primaria completa,F_EDUCACIONPADRE_Primaria incompleta,F_EDUCACIONPADRE_Secundaria (Bachillerato) completa,F_EDUCACIONPADRE_Secundaria (Bachillerato) incompleta,F_EDUCACIONPADRE_Técnica o tecnológica completa,F_EDUCACIONPADRE_Técnica o tecnológica incompleta
0,904256,0.933333,-0.362385,BOGOTÁ,-0.01033,-0.333333,0.000000,NaN,Si,Si,...,False,False,False,False,False,False,False,False,False,True
1,645256,0.933333,-0.467890,ATLANTICO,-0.01033,-1.000000,0.000000,NaN,Si,No,...,False,False,False,False,False,False,False,False,True,False
2,308367,0.333333,0.736239,BOGOTÁ,-0.01033,1.000000,0.000000,NaN,Si,No,...,False,False,False,False,False,False,True,False,False,False
3,470353,-0.200000,-0.970183,SANTANDER,-0.01033,-1.000000,0.333333,NaN,Si,No,...,False,False,True,False,False,False,False,False,False,False
4,989032,0.933333,0.919725,ANTIOQUIA,-0.01033,0.700000,0.000000,NaN,Si,Si,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80822,401241,0.333333,-0.176606,ANTIOQUIA,-0.01033,-0.333333,0.000000,NaN,Si,No,...,False,False,False,True,False,False,False,False,False,False
80823,610107,0.333333,0.919725,SANTANDER,-0.01033,-1.000000,-0.333333,NaN,Si,Si,...,False,False,False,False,False,False,False,False,True,False
80824,850931,-1.000000,-0.467890,QUINDIO,-0.01033,-1.000000,0.000000,NaN,Si,Si,...,False,False,False,False,False,False,False,False,False,False
80825,298250,-1.000000,-0.176606,BOYACA,-0.01033,0.261319,-0.333333,NaN,NaN,NaN,...,False,True,False,False,False,False,False,False,False,False


In [17]:
#Validacion diccionarios
import pandas as pd

# Crear diccionario para almacenar información de cada departamento
departments_info = {}

# Ordenar departamentos por estrato promedio
dept_estrato = data.groupby('E_PRGM_DEPARTAMENTO')['F_ESTRATOVIVIENDA'].mean().sort_values()
departments = dept_estrato.index.tolist()

for department in departments:
    dept_data = data[data['E_PRGM_DEPARTAMENTO'] == department].copy()

    # Eliminar filas con etiqueta NaN
    dept_data = dept_data.dropna(subset=['RENDIMIENTO_GLOBAL'])

    # Revisar si hay suficiente data
    if len(dept_data) < 2:
        status = f"NO ENTRENADO (solo {len(dept_data)} muestra)"
        departments_info[department] = {'data': dept_data, 'status': status}
        print(f"---{department}---\n{status}")
        continue

    # Revisar si hay al menos 2 clases
    y_train = dept_data['RENDIMIENTO_GLOBAL']
    if y_train.nunique() < 2:
        status = "NO ENTRENADO (solo una clase)"
        departments_info[department] = {'data': dept_data, 'status': status}
        print(f"---{department}---\n{status}")
        continue

    # Si pasa todas las validaciones
    departments_info[department] = {'data': dept_data, 'status': 'OK'}


---VAUPES---
NO ENTRENADO (solo 1 muestra)
---AMAZONAS---
NO ENTRENADO (solo una clase)


In [18]:
#Entrenamiento
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

models_by_department = {}

for department, info in departments_info.items():

    print(f"---{department}---")

    if info['status'] != 'OK':
        print(f"Reporte de clasificación para {department}: {info['status']}")
        continue

    dept_data = info['data']

    X_train = dept_data.drop(columns=['RENDIMIENTO_GLOBAL', 'ID', 'E_PRGM_DEPARTAMENTO'])
    y_train = dept_data['RENDIMIENTO_GLOBAL']

    # Columnas categóricas y numéricas
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    numeric_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()

    # Preprocesador con imputación
    preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), categorical_cols),
            ('numeric', SimpleImputer(strategy='mean'), numeric_cols)
        ]
    )

    # Modelo LightGBM
    lgbm_model = LGBMClassifier(
        random_state=71,
        objective='multiclass',
        num_class=4,
        boosting_type='gbdt',
        learning_rate=0.05,
        n_estimators=500,
        max_depth=10,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        reg_alpha=0.1,
        min_child_weight=3,
        min_split_gain=0.1,
        importance_type='gain',
        verbosity=-1
    )

    # Pipeline final
    model = Pipeline([
        ('preprocess', preprocessor),
        ('lgbm', lgbm_model)
    ])

    # Entrenamiento
    model.fit(X_train, y_train)

    # Predicción sobre train
    y_pred = model.predict(X_train)
    accuracy = accuracy_score(y_train, y_pred)

    print(f"Reporte de clasificación para {department}: {accuracy}")

    # Guardar modelo
    models_by_department[department] = model


---VAUPES---
Reporte de clasificación para VAUPES: NO ENTRENADO (solo 1 muestra)
---PUTUMAYO---
Reporte de clasificación para PUTUMAYO: 0.8297872340425532
---GUAVIARE---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/us

Reporte de clasificación para GUAVIARE: 0.75
---CHOCO---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Reporte de clasificación para CHOCO: 0.9648760330578512
---LA GUAJIRA---


...

Reporte de clasificación para CALDAS: 0.9749631811487481
---CUNDINAMARCA---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Reporte de clasificación para CUNDINAMARCA: 0.9474009900990099
---ANTIOQUIA---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Reporte de clasificación para ANTIOQUIA: 0.7941873647325569


In [19]:
#Carga de datos para el Predict


test = pd.read_csv('test.csv')
test_data = test.copy()

In [20]:
# --- IMPUTACIÓN CATEGÓRICA SEGURA ---

# 1. Obtener las columnas con las que se entrenó el imputador
cols_train = list(cat_imputer.feature_names_in_)

# 2. Crear en test_data cualquier columna que falte
for col in cols_train:
    if col not in test_data.columns:
        test_data[col] = None

# 3. Asegurar el mismo orden de columnas
test_data_subset = test_data[cols_train]

# 4. Aplicar transform a test (sin fit)
test_data[cols_train] = cat_imputer.transform(test_data_subset)

In [21]:
# 1. Obtener las columnas con las que el imputador fue entrenado
cols_train = list(cat_imputer.feature_names_in_)

# 2. Crear columnas faltantes en test_data
for col in cols_train:
    if col not in test_data.columns:
        test_data[col] = None

# 3. Tomar test_data únicamente con las columnas esperadas, en el orden correcto
test_cat = test_data[cols_train]

# 4. Aplicar transform (solo transform, sin fit)
test_data[cols_train] = cat_imputer.transform(test_cat)



In [22]:
from sklearn.impute import SimpleImputer

# 1. Imputación manual para columnas numéricas completamente vacías
empty_numeric = ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']

for col in empty_numeric:
    if test_data[col].isna().all():
        print(f"La columna {col} está completamente vacía, imputando con 0.")
        test_data[col] = 0


# 2. Imputación manual para columnas categóricas completamente vacías
empty_categorical = ['F_TIENEINTERNET']

for col in empty_categorical:
    if test_data[col].isna().all():
        print(f"La columna categórica {col} está completamente vacía, imputando con 'Desconocido'.")
        test_data[col] = 'Desconocido'


# 3. Ahora, imputar SOLO las columnas categóricas que sí tienen datos
cat_columns = ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET']

# Filtrar columnas que NO están completamente vacías
cols_with_data = [col for col in cat_columns if not test_data[col].isna().all()]

print("Columnas categóricas con datos para imputar:", cols_with_data)

if len(cols_with_data) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    test_data[cols_with_data] = cat_imputer.fit_transform(test_data[cols_with_data])
else:
    print("No hay columnas categóricas con datos suficientes para imputar.")




Columnas categóricas con datos para imputar: ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET']


In [23]:
# Eliminar columna innecesaria
if 'Unnamed: 0' in test_data.columns:
    test_data = test_data.drop('Unnamed: 0', axis=1)

# Transformar valores inconsistentes en columnas específicas
test_data['E_VALORMATRICULAUNIVERSIDAD'] = test_data['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)
test_data['E_HORASSEMANATRABAJA'] = test_data['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

# Imputar valores faltantes en columnas numéricas
for col in ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']:
    if test_data[col].isna().all():
        print(f"La columna {col} está completamente vacía, imputando con 0.")
        test_data[col] = 0

# Escalar columnas numéricas
num_columns = test_data.select_dtypes(include=['number']).columns
num_columns = [col for col in num_columns if col not in ['ID']]
test_data[num_columns] = scaler.fit_transform(test_data[num_columns])


In [24]:
# Eliminar columna innecesaria
if 'Unnamed: 0' in test_data.columns:
    test_data = test_data.drop('Unnamed: 0', axis=1)

# Transformar valores inconsistentes en columnas específicas
test_data['E_VALORMATRICULAUNIVERSIDAD'] = test_data['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)
test_data['E_HORASSEMANATRABAJA'] = test_data['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

# Imputar valores faltantes en columnas numéricas
for col in ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']:
    if test_data[col].isna().all():
        print(f"La columna {col} está completamente vacía, imputando con 0.")
        test_data[col] = 0

# Escalar columnas numéricas
num_columns = test_data.select_dtypes(include=['number']).columns
num_columns = [col for col in num_columns if col not in ['ID']]
test_data[num_columns] = scaler.fit_transform(test_data[num_columns])


La columna E_VALORMATRICULAUNIVERSIDAD está completamente vacía, imputando con 0.
La columna E_HORASSEMANATRABAJA está completamente vacía, imputando con 0.


In [25]:
test_data

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,F_TIENEAUTOMOVIL,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,550236,-1.000000,TRABAJO SOCIAL,BOLIVAR,-1.0,-1.0,Estrato 3,Si,Técnica o tecnológica completa,Si,No,N,Si,Si,Si,Primaria completa,-0.010558,-0.095041,0.968944,0.492447
1,98545,0.333333,ADMINISTRACION COMERCIAL Y DE MERCADEO,ANTIOQUIA,-1.0,-1.0,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Técnica o tecnológica completa,-0.315234,0.169421,0.838509,0.957704
2,499179,0.933333,INGENIERIA MECATRONICA,BOGOTÁ,-1.0,-1.0,Estrato 3,Si,Secundaria (Bachillerato) incompleta,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,-0.140271,-0.057851,0.826087,0.492447
3,782980,-0.200000,CONTADURIA PUBLICA,SUCRE,-1.0,-1.0,Estrato 1,No,Primaria incompleta,Si,No,N,No,No,No,Primaria incompleta,-0.517345,0.685950,0.347826,0.776435
4,785185,0.933333,ADMINISTRACION DE EMPRESAS,ATLANTICO,-1.0,-1.0,Estrato 2,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Secundaria (Bachillerato) completa,-0.369532,0.169421,0.900621,0.728097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296781,496981,-0.200000,ADMINISTRACION DE EMPRESAS,BOGOTÁ,-1.0,-1.0,Estrato 1,Si,Primaria incompleta,Si,Si,N,Si,Si,Si,Primaria incompleta,-0.493213,0.694215,0.459627,0.812689
296782,209415,-1.000000,DERECHO,META,-1.0,-1.0,Estrato 4,Si,Educación profesional completa,Si,No,N,No,Si,Si,Educación profesional completa,0.420814,-0.239669,0.639752,0.166163
296783,239074,0.933333,DERECHO,BOGOTÁ,-1.0,-1.0,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,No,N,No,Si,Si,Educación profesional completa,-0.119155,0.028926,0.714286,0.546828
296784,963852,-0.200000,INGENIERIA AERONAUTICA,ANTIOQUIA,-1.0,-1.0,Estrato 3,Si,Educación profesional completa,Si,No,N,No,Si,Si,Educación profesional completa,-0.079940,-0.095041,0.925466,0.570997


In [26]:
results = []

# Create a copy of the original test data to apply transformations
processed_test_data = test.copy()

# --- Reapply all preprocessing steps as done for training data ---

# 1. Apply custom conversion functions for matrícula and horas trabajadas
processed_test_data['E_VALORMATRICULAUNIVERSIDAD'] = processed_test_data['E_VALORMATRICULAUNIVERSIDAD'].apply(valor_matricula)
processed_test_data['E_HORASSEMANATRABAJA'] = processed_test_data['E_HORASSEMANATRABAJA'].apply(horas_trabajadas)

# 2. Impute numerical columns using the *fitted* num_imputer from training
num_cols_for_imputation = ['E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']
# Ensure columns exist before transforming
for col in num_cols_for_imputation:
    if col not in processed_test_data.columns:
        processed_test_data[col] = np.nan
processed_test_data[num_cols_for_imputation] = preprocessing_objects['num_imputer'].transform(
    processed_test_data[num_cols_for_imputation]
)

# 3. Impute categorical columns using the *fitted* cat_imputer from training
cat_cols_for_imputation_from_train = ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'E_PAGOMATRICULAPROPIO']
for col in cat_cols_for_imputation_from_train:
    if col not in processed_test_data.columns:
        processed_test_data[col] = np.nan
processed_test_data[cat_cols_for_imputation_from_train] = preprocessing_objects['cat_imputer'].transform(
    processed_test_data[cat_cols_for_imputation_from_train]
)

# 4. Map binary columns (F_TIENEINTERNET, E_PAGOMATRICULAPROPIO)
processed_test_data = map_binary_columns(processed_test_data, binary_columns)

# 5. Map F_ESTRATOVIVIENDA to numerical values
processed_test_data['F_ESTRATOVIVIENDA'] = processed_test_data['F_ESTRATOVIVIENDA'].replace(estrato_dict)
processed_test_data['F_ESTRATOVIVIENDA'] = pd.to_numeric(processed_test_data['F_ESTRATOVIVIENDA'], errors='coerce')

# 6. Apply Label Encoding for E_PRGM_ACADEMICO (using fitted encoder)
col_le = 'E_PRGM_ACADEMICO'
le = preprocessing_objects['label_encoders'][col_le]
# Handle unseen categories by mapping to a known class (e.g., the first class seen during fit)
processed_test_data[col_le] = processed_test_data[col_le].apply(lambda s: s if s in le.classes_ else le.classes_[0])
processed_test_data[col_le] = le.transform(processed_test_data[col_le])

# 7. One-Hot Encoding for F_EDUCACIONMADRE and F_EDUCACIONPADRE
def apply_onehot_encoding_test_robust(df, original_cols_to_encode, fitted_onehot_encoders, fill_value='No Aplica'):
    df_copy = df.copy()
    for col in original_cols_to_encode:
        df_copy[col].fillna(fill_value, inplace=True)
        train_unique_values = fitted_onehot_encoders[col] # Get unique values from fitted encoder

        dummies = pd.get_dummies(df_copy[col], prefix=col, dtype=int)

        # Reindex to match the columns from training, filling missing with 0
        expected_cols = [f"{col}_{val}" for val in train_unique_values]
        dummies = dummies.reindex(columns=expected_cols, fill_value=0)

        df_copy = pd.concat([df_copy, dummies], axis=1)
        df_copy.drop(col, axis=1, inplace=True)
    return df_copy

processed_test_data = apply_onehot_encoding_test_robust(
    processed_test_data,
    columns_to_encode, # This is ['F_EDUCACIONMADRE','F_EDUCACIONPADRE']
    preprocessing_objects['onehot_encoders']
)

# 8. Apply Scaling for numerical columns as done in training.
# Note: The original training code overwrote the 'scaler' object. To replicate the *effect* of scaling as in training,
# we will apply new MinMaxScaler instances to the identified columns. This is a workaround as the original fitted scalers were not saved.
# For proper production, these fitted scalers should be saved and loaded.

# Identify all columns that were scaled in the training 'data' DataFrame
# These include the original numeric columns, E_PRGM_ACADEMICO, and F_ESTRATOVIVIENDA.
# The one-hot encoded columns are also numeric and handled by the ColumnTransformer's numeric imputer.

# List of columns that should be scaled (non-object columns excluding 'ID' and 'RENDIMIENTO_GLOBAL' if present)
scaling_target_columns = [col for col in processed_test_data.columns
                          if processed_test_data[col].dtype != 'object' and col not in ['ID', 'RENDIMIENTO_GLOBAL']]

# Apply a fresh MinMaxScaler to each of these columns. This scales based on test data's min/max.
# This is an approximation due to the original scaler not being saved. For robust deployment, save fitted scalers.
for col in scaling_target_columns:
    scaler_temp = MinMaxScaler(feature_range=(-1, 1))
    processed_test_data[col] = scaler_temp.fit_transform(processed_test_data[[col]])


# --- End of preprocessing for test data ---

# Now, proceed with predictions using the fully preprocessed test data
for department, model in models_by_department.items():
    print(f"Realizando predicciones para el departamento: {department}")

    # Filter the processed test data for the current department
    test_dept_data = processed_test_data[processed_test_data['E_PRGM_DEPARTAMENTO'] == department]

    if test_dept_data.empty:
        continue

    # Separate features for prediction
    X_test = test_dept_data.drop(columns=['ID', 'E_PRGM_DEPARTAMENTO', 'RENDIMIENTO_GLOBAL'], errors='ignore')

    y_test_pred = model.predict(X_test)

    # Convert numerical predictions back to original labels
    label_dict = {v: k for k, v in rendimiento_mapping.items()}
    y_test_pred_labels = [label_dict[int(pred)] for pred in y_test_pred]

    # Store results
    results.extend(zip(test_dept_data['ID'], y_test_pred_labels))

# Handle departments not seen during training
unseen_departments = set(test['E_PRGM_DEPARTAMENTO'].unique()) - set(models_by_department.keys())
for department in unseen_departments:
    print(f"No hay datos de entrenamiento para el departamento: {department}. Asignando predicciones globales.")
    # Get original test data for this department to extract IDs
    original_test_dept_data = test[test['E_PRGM_DEPARTAMENTO'] == department]
    y_test_pred_labels = ['medio-bajo'] * len(original_test_dept_data)  # Default prediction
    results.extend(zip(original_test_dept_data['ID'], y_test_pred_labels))

# Create submission file
submission = pd.DataFrame(results, columns=['ID', 'RENDIMIENTO_GLOBAL'])
submission.to_csv('submission-IA.csv', index=False)
print("Archivo de predicciones creado: 'submission.csv'")

/tmp/ipython-input-2665002775.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  processed_test_data['F_ESTRATOVIVIENDA'] = processed_test_data['F_ESTRATOVIVIENDA'].replace(estrato_dict)
/tmp/ipython-input-2665002775.py:49: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy[col].fillna(fill_val

Realizando predicciones para el departamento: PUTUMAYO
Realizando predicciones para el departamento: GUAVIARE
Realizando predicciones para el departamento: CHOCO
Realizando predicciones para el departamento: LA GUAJIRA


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: 

Realizando predicciones para el departamento: CAQUETA
Realizando predicciones para el departamento: ARAUCA
Realizando predicciones para el departamento: CORDOBA


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

Realizando predicciones para el departamento: CUNDINAMARCA


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Realizando predicciones para el departamento: ANTIOQUIA


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['F_TIENEINTERNET' 'E_PAGOMATRICULAPROPIO']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


No hay datos de entrenamiento para el departamento: AMAZONAS. Asignando predicciones globales.
No hay datos de entrenamiento para el departamento: SAN ANDRES. Asignando predicciones globales.
No hay datos de entrenamiento para el departamento: VAUPES. Asignando predicciones globales.
Archivo de predicciones creado: 'submission.csv'


In [27]:
import pandas as pd

# Cargar archivo
sub = pd.read_csv('submission-IA.csv')

# Conteo de categorías en la columna
conteo = sub['RENDIMIENTO_GLOBAL'].value_counts()

print(conteo)

RENDIMIENTO_GLOBAL
alto          83223
bajo          78851
medio-bajo    67735
medio-alto    66977
Name: count, dtype: int64
